In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e3/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e3/train.csv
/kaggle/input/competitions/playground-series-s6e3/test.csv


In [3]:
train=pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/train.csv')

In [4]:
test=pd.read_csv('/kaggle/input/competitions/playground-series-s6e3/test.csv')

In [5]:
train.head(5)

,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [6]:
train.shape

(594194, 21)

In [8]:
train.isnull().sum()

id                  0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [9]:
train.describe()

,id,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,594194.000000,594194.000000,594194.000000,594194.000000,594194.000000
mean,297096.500000,0.114102,36.577258,65.866223,2494.377057
std,171529.177262,0.317936,25.061922,31.067444,2353.916710
min,0.000000,0.000000,1.000000,18.250000,18.800000
25%,148548.250000,0.000000,12.000000,29.900000,639.650000
50%,297096.500000,0.000000,35.000000,74.100000,1433.650000
75%,445644.750000,0.000000,62.000000,90.800000,4263.800000
max,594193.000000,1.000000,72.000000,118.750000,8684.800000


In [14]:
train['Churn'].value_counts(normalize=True)

Churn
No     0.774792
Yes    0.225208
Name: proportion, dtype: float64

In training data, the classes are imbalanced, they are of 77% No and 22% yes

In [15]:
train.dtypes

id                    int64
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

In [16]:
train.columns

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [19]:
train.columns

Index(['id', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [25]:
numeric_cols = train.select_dtypes(include='number').columns.tolist()
numeric_cols

['id', 'SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

In [22]:
cat_cols = train.select_dtypes(include='object').columns.tolist()
cat_cols

['gender',
 'Partner',
 'Dependents',
 'PhoneService',
 'MultipleLines',
 'InternetService',
 'OnlineSecurity',
 'OnlineBackup',
 'DeviceProtection',
 'TechSupport',
 'StreamingTV',
 'StreamingMovies',
 'Contract',
 'PaperlessBilling',
 'PaymentMethod',
 'Churn']

In [ ]:
# ============================================
# PART A — Central Tendency
# ============================================
print("=" * 60)
print("PART A — Mean, Median, Mode + Skewness")
print("=" * 60)

for col in continuous_cols:
    mean   = train[col].mean()
    median = train[col].median()
    mode   = train[col].mode()[0]
    skew   = train[col].skew()
    
    # Interpret skewness
    if skew > 1:
        skew_label = 'highly right skewed ⚠️'
    elif skew > 0.5:
        skew_label = 'moderately right skewed'
    elif skew < -1:
        skew_label = 'highly left skewed ⚠️'
    elif skew < -0.5:
        skew_label = 'moderately left skewed'
    else:
        skew_label = 'roughly symmetric ✅'
    
    print(f"Column: {col}")
    print(f"  Mean: {mean:.2f}")
    print(f"  Median: {median:.2f}")
    print(f"  Mode: {mode:.2f}")
    print(f"  Skewness:{skew:.2f} → {skew_label}")
    
    # Flag if mean and median differ a lot
    diff_pct = abs(mean - median) / median * 100
    if diff_pct > 10:
        print(f"Mean and Median differ by {diff_pct:.1f}%")
        print(f"outliers or skewness present")
    print()


In [ ]:
for col in continuous_cols:
    Q1  = train[col].quantile(0.25)
    Q3  = train[col].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = train[
        (train[col] < lower_bound) | 
        (train[col] > upper_bound)
    ]
    
    outlier_pct = len(outliers) / len(train) * 100
    
    print(f"Column: {col}")
    print(f"  Q1:          {Q1:.2f}")
    print(f"  Q3:          {Q3:.2f}")
    print(f"  IQR:         {IQR:.2f}")
    print(f"  Lower bound: {lower_bound:.2f}")
    print(f"  Upper bound: {upper_bound:.2f}")
    print(f"  Outliers:    {len(outliers):,} ({outlier_pct:.1f}%)")
    
    if outlier_pct > 5:
        print(f"  ⚠️  High outlier rate! Consider capping or log transform")
    elif outlier_pct > 0:
        print(f"  ℹ️  Some outliers present, monitor")
    else:
        print(f"  ✅ No outliers detected")
    print()

In [ ]:
fig, axes = plt.subplots(
    len(continuous_cols), 3, 
    figsize=(15, 4 * len(continuous_cols))
)

for i, col in enumerate(continuous_cols):
    
    # Plot 1 — Histogram with KDE
    axes[i,0].hist(train[col], bins=30, 
                   color='steelblue', edgecolor='white',
                   alpha=0.7)
    axes[i,0].axvline(train[col].mean(), 
                      color='red', linestyle='--', 
                      label='Mean')
    axes[i,0].axvline(train[col].median(), 
                      color='green', linestyle='--', 
                      label='Median')
    axes[i,0].set_title(f'{col} — Distribution')
    axes[i,0].set_xlabel(col)
    axes[i,0].set_ylabel('Count')
    axes[i,0].legend()
    
    # Plot 2 — Boxplot (shows outliers visually)
    axes[i,1].boxplot(train[col].dropna(), 
                      vert=True,
                      patch_artist=True,
                      boxprops=dict(facecolor='lightblue'))
    axes[i,1].set_title(f'{col} — Boxplot')
    axes[i,1].set_ylabel(col)
    
    # Plot 3 — Churners vs Non Churners
    churn_yes = train[train['Churn']==1][col]
    churn_no  = train[train['Churn']==0][col]
    
    axes[i,2].hist(churn_no,  bins=30, alpha=0.6, 
                   color='steelblue', label='No Churn',
                   density=True)
    axes[i,2].hist(churn_yes, bins=30, alpha=0.6, 
                   color='tomato',    label='Churn',
                   density=True)
    axes[i,2].set_title(f'{col} — Churn vs No Churn')
    axes[i,2].set_xlabel(col)
    axes[i,2].set_ylabel('Density')
    axes[i,2].legend()

plt.tight_layout()
plt.show()